In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
    .appName("Carga Delta")\
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")\
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")\
    .getOrCreate()


In [0]:
dbutils.fs.mkdirs("/Volumes/vendas_pecas/default/my/Ouro")

In [0]:
silver_path = "/Volumes/vendas_pecas/default/my/prata"
silver_table = "vendas_pecas.camada_prata.tb_prata"
gold_path = "/Volumes/vendas_pecas/default/my/Ouro"

In [0]:
df_ouro = spark.table(silver_table)

In [0]:
df_ouro.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("vendas_pecas.camada_ouro.Fato_Vendas")

##Tabelas Dimensão

In [0]:
%sql
CREATE OR REPLACE TABLE tb_dim_produtos AS
SELECT
  ROW_NUMBER() OVER (ORDER BY produto_peca_sem_acento) as produto_id,
  produto_peca_sem_acento
FROM 
  (SELECT DISTINCT (produto_peca_sem_acento) FROM vendas_pecas.camada_ouro.fato_vendas)
ORDER BY (produto_peca_sem_acento)
       

In [0]:
%sql

CREATE OR REPLACE TABLE tb_dim_cliente AS
SELECT
  cliente_id,
  cliente_nome
FROM
  (SELECT DISTINCT
  cliente_id,
  cliente_nome FROM vendas_pecas.camada_ouro.fato_vendas)

  

In [0]:
%sql

CREATE OR REPLACE TABLE tb_dim_marca_carro AS
SELECT 
  row_number() OVER (ORDER BY marca_carro) AS marca_carro_id,
  marca_carro
FROM
  (SELECT DISTINCT
  marca_carro FROM vendas_pecas.camada_ouro.fato_vendas)

In [0]:
%sql 
CREATE OR REPLACE TABLE tb_dim_modelo_carro AS
SELECT
  ROW_NUMBER() OVER (ORDER BY modelo_carro) AS modelo_carro_id,
  modelo_carro
FROM
  (SELECT DISTINCT modelo_carro FROM vendas_pecas.camada_ouro.fato_vendas)

In [0]:
%sql

CREATE  OR REPLACE TABLE tb_modelo_marca AS
SELECT
  m.modelo_carro_id,
  ma.marca_carro_id
FROM (
    SELECT DISTINCT modelo_carro, marca_carro FROM vendas_pecas.camada_ouro.fato_vendas
) AS c
JOIN
  tb_dim_modelo_carro AS m
ON
  c.modelo_carro = m.modelo_carro
JOIN
  tb_dim_marca_carro AS ma
ON
  c.marca_carro = ma.marca_carro

     
  

In [0]:
%sql
CREATE OR REPLACE TABLE tb_dim_carro AS
SELECT
  ROW_NUMBER() OVER (order by c.modelo_carro) AS carro_id,
  modelo_carro_id,
  marca_carro_id
FROM (SELECT DISTINCT modelo_carro, marca_carro FROM vendas_pecas.camada_ouro.fato_vendas) c
JOIN tb_dim_modelo_carro mc
ON c.modelo_carro = mc.modelo_carro
JOIN tb_dim_marca_carro m
ON c.marca_carro = m.marca_carro

In [0]:
%sql

CREATE OR REPLACE TABLE vendas_pecas.camada_ouro.tb_dim_vendedor AS
SELECT DISTINCT
  vendedor_id,
  vendedor_nome
FROM
   vendas_pecas.camada_ouro.fato_vendas

In [0]:
%sql
CREATE OR REPLACE TABLE vendas_pecas.camada_ouro.tb_dim_loja AS
SELECT DISTINCT
  loja_id,
  loja_nome_sem_acento,
  grupo_loja_sem_acento
FROM  vendas_pecas.camada_ouro.fato_vendas
order by grupo_loja_sem_acento

In [0]:
%sql

CREATE OR REPLACE TABLE vendas_pecas.camada_ouro.tb_dim_data_venda AS
SELECT
  ROW_NUMBER() OVER (ORDER BY data_venda) AS data_venda_id,
  data_venda, ano_venda,  mes_venda, DAY(data_venda) as dia_venda
FROM (SELECT DISTINCT data_venda, ano_venda, mes_venda FROM vendas_pecas.camada_ouro.fato_vendas)



###Tabela Fato

In [0]:
%sql

use vendas_pecas.camada_ouro;

CREATE OR REPLACE TABLE vendas_pecas.camada_ouro.tb_fato_vendas AS
SELECT
  fv.IdNotaFiscal,
  fv.valor_unitario,
  fv.quantidade,
  fv.custo_unitario,
  dcl.cliente_id,
  dc.carro_id,
  ddv.data_venda_id,
  dl.loja_id,
  dpd.produto_id,
  dv.vendedor_id
FROM vendas_pecas.camada_ouro.fato_vendas fv
LEFT JOIN tb_dim_cliente dcl ON fv.cliente_id = dcl.cliente_id
LEFT JOIN tb_dim_data_venda ddv ON fv.data_venda = ddv.data_venda
LEFT JOIN tb_dim_loja dl ON fv.loja_nome_sem_acento = dl.loja_nome_sem_acento
LEFT JOIN tb_dim_produtos dpd ON fv.produto_peca_sem_acento = dpd.produto_peca_sem_acento
LEFT JOIN tb_dim_vendedor dv ON fv.vendedor_nome = dv.vendedor_nome
LEFT JOIN tb_dim_modelo_carro md  ON fv.modelo_carro = md.modelo_carro 
LEFT JOIN tb_dim_marca_carro mc ON fv.marca_carro = mc.marca_carro
LEFT JOIN tb_dim_carro dc ON dc.modelo_carro_id = md.modelo_carro_id and dc.marca_carro_id = mc.marca_carro_id

order by IdNotaFiscal 
                                            

In [0]:
%sql
select distinct fv.cliente_id, md.modelo_carro, mc.marca_carro, dc.carro_id from tb_dim_carro dc
LEFT JOIN tb_dim_modelo_carro md  ON dc.modelo_carro_id = md.modelo_carro_id
LEFT JOIN tb_dim_marca_carro mc ON mc.marca_carro_id = dc.marca_carro_id
LEFT JOIN fato_vendas fv ON fv.modelo_carro = md.modelo_carro and fv.marca_carro = mc.marca_carro

where fv.cliente_id = "8301661318"

In [0]:
%sql

select distinct dc.carro_id, fv.cliente_id, md.modelo_carro, mc.marca_carro from fato_vendas fv
left JOIN tb_dim_modelo_carro md  ON fv.modelo_carro = md.modelo_carro 
left JOIN tb_dim_marca_carro mc ON fv.marca_carro = mc.marca_carro
LEFT JOIN tb_dim_carro dc ON dc.modelo_carro_id = md.modelo_carro_id and dc.marca_carro_id = mc.marca_carro_id
where cliente_id = "8301661318"


In [0]:
%sql 
use vendas_pecas.camada_ouro;

select distinct tfv.IdNotaFiscal, tfv.cliente_id, dc.cliente_nome, md.modelo_carro, mc.marca_carro  from tb_fato_vendas tfv
join tb_dim_cliente dc on tfv.cliente_id = dc.cliente_id
join tb_dim_carro dcr on tfv.carro_id = dcr.carro_id
join tb_dim_modelo_carro md on dcr.modelo_carro_id = md.modelo_carro_id
join tb_dim_marca_carro mc on dcr.marca_carro_id = mc.marca_carro_id
where tfv.IdNotaFiscal like "NF150000"



In [0]:
%sql
use vendas_pecas.camada_ouro;

select distinct IdNotaFiscal, data_venda, dcl.cliente_id, cliente_nome,marca_carro, modelo_carro, valor_unitario, quantidade, custo_unitario, dlj.loja_id, loja_nome_sem_acento, grupo_loja_sem_acento, dvr.vendedor_id, vendedor_nome, ano_venda, mes_venda, produto_peca_sem_acento from tb_fato_vendas fv
RIGHT JOIN tb_dim_data_venda dtv ON fv.data_venda_id = dtv.data_venda_id
RIGHT JOIN tb_dim_cliente dcl ON fv.cliente_id = dcl.cliente_id 
RIGHT JOIN tb_dim_carro dcr on fv.carro_id = dcr.carro_id
RIGHT JOIN tb_dim_modelo_carro md on dcr.modelo_carro_id = md.modelo_carro_id
RIGHT JOIN tb_dim_marca_carro mc on dcr.marca_carro_id = mc.marca_carro_id
RIGHT JOIN tb_dim_produtos dpd ON fv.produto_id = dpd.produto_id
RIGHT JOIN tb_dim_loja dlj ON fv.loja_id = dlj.loja_id
RIGHT JOIN tb_dim_vendedor dvr ON fv.vendedor_id = dvr.vendedor_id
where fv.cliente_id = "8301661318";




In [0]:
%sql
use vendas_pecas.camada_ouro;

(select any_value(ano_venda) as ano_venda, mes_venda, concat('R$ ', format_number(sum(valor_unitario * quantidade),'0,000.00')) as Faturamento_Mensal 
from tb_fato_vendas fv
join tb_dim_data_venda ddt on fv.data_venda_id = ddt.data_venda_id
where fv.valor_unitario > 0 and fv.quantidade Is not Null and ano_venda == 2024
group by ddt.mes_venda
order by ddt.mes_venda)
UNION ALL
(select any_value(ano_venda), mes_venda, concat('R$ ', format_number(sum(valor_unitario * quantidade),'0,000.00')) as Faturamento_Mensal 
from tb_fato_vendas fv
join tb_dim_data_venda ddt on fv.data_venda_id = ddt.data_venda_id
where fv.valor_unitario > 0 and fv.quantidade Is not Null and ano_venda == 2025
group by ddt.mes_venda
order by ddt.mes_venda)


In [0]:
%sql
use vendas_pecas.camada_ouro;

select any_value(ano_venda), row_number() OVER (PARTITION BY ddt.ano_venda ORDER BY ddt.mes_venda, ddt.ano_venda), concat('R$ ', format_number(sum(valor_unitario * quantidade),'0,000.00')) as Faturamento_Mensal 
from tb_fato_vendas fv
join tb_dim_data_venda ddt on fv.data_venda_id = ddt.data_venda_id
where fv.valor_unitario > 0 and fv.quantidade Is not Null
group by ddt.mes_venda, ddt.ano_venda
order by ddt.mes_venda

In [0]:
%sql

select 
    ano_venda, 
    mes_venda, 
    concat('R$ ', format_number(sum(valor_unitario * quantidade),'0,000.00')) as Faturamento_Mensal 
from 
  tb_fato_vendas fv
join 
  tb_dim_data_venda ddt on fv.data_venda_id = ddt.data_venda_id
where 
  fv.valor_unitario > 0 and fv.quantidade Is not Null 
group by 
  ROLLUP (ano_venda, mes_venda)
order by 
  CASE WHEN ddt.ano_venda IS NULL THEN 1 ELSE 0 END, ano_venda,
  CASE WHEN ddt.mes_venda IS NULL THEN 1 ELSE 0 END, mes_venda

In [0]:
%sql
use vendas_pecas.camada_ouro;

SELECT 
  grupo_loja_sem_acento as grupo_loja,
  loja_nome_sem_acento as loja_nome,
  concat('R$ ', format_number(sum(valor_unitario * quantidade),'0,000.00')) as Faturamento 
FROM tb_fato_vendas fv
JOIN tb_dim_loja dl ON fv.loja_id = dl.loja_id
WHERE fv.valor_unitario > 0 AND fv.quantidade IS NOT NULL
GROUP BY loja_nome_sem_acento, grupo_loja
ORDER BY loja_nome_sem_acento
   

In [0]:
%sql
use vendas_pecas.camada_ouro;

SELECT 
  loja_nome_sem_acento as loja_nome,
  vendedor_nome,
  dv.vendedor_id,
  concat('R$ ', format_number(sum(valor_unitario * quantidade),'0,000.00')) as Faturamento 
FROM tb_fato_vendas fv
JOIN tb_dim_loja dl ON fv.loja_id = dl.loja_id
JOIN tb_dim_vendedor dv ON fv.vendedor_id = dv.vendedor_id 
WHERE fv.valor_unitario > 0 AND fv.quantidade IS NOT NULL
GROUP BY loja_nome,vendedor_nome, dv.vendedor_id
ORDER BY loja_nome, vendedor_id
   